# AMR 스마트 물류창고 출고 지연 예측

## 1. 라이브러리 import

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

print(f"pandas: {pd.__version__}")
print(f"numpy:  {np.__version__}")
print(f"lightgbm: {lgb.__version__}")

pandas: 2.3.3
numpy:  2.3.4
lightgbm: 4.6.0


## 2. 데이터 로드 및 기본 정보 확인

In [2]:
# 데이터 경로 (필요시 수정)
DATA_DIR = '.'  # 현재 폴더에 csv가 있다고 가정

train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')
layout = pd.read_csv(f'{DATA_DIR}/layout_info.csv')

target_col = 'avg_delay_minutes_next_30m'

print(f"train shape: {train.shape}")
print(f"test shape:  {test.shape}")
print(f"layout shape: {layout.shape}")
print(f"\ntrain target stats: mean={train[target_col].mean():.2f}, "
      f"std={train[target_col].std():.2f}, max={train[target_col].max():.2f}")
print(f"\ntrain layouts: {train['layout_id'].nunique()}, "
      f"test layouts: {test['layout_id'].nunique()}")
print(f"신규 layout (test에만): "
      f"{len(set(test['layout_id'].unique()) - set(train['layout_id'].unique()))}")

train shape: (250000, 94)
test shape:  (50000, 93)
layout shape: (300, 15)

train target stats: mean=18.96, std=27.35, max=715.86

train layouts: 250, test layouts: 100
신규 layout (test에만): 50


## 3. 전처리: layout_info join + slot 위치 추가`slot_idx`는 시나리오 안에서 슬롯의 위치 (0~24)를 나타냅니다. 시간 진화 분석에서 핵심 변수입니다.

In [3]:
# layout_info join
train = train.merge(layout, on='layout_id', how='left')
test = test.merge(layout, on='layout_id', how='left')

# 슬롯 위치 추가 (시나리오 안의 시간 인덱스)
def add_slot_idx(df):
    df = df.sort_values(['scenario_id', 'ID']).reset_index(drop=True)
    df['slot_idx'] = df.groupby('scenario_id').cumcount()
    return df

train = add_slot_idx(train)
test = add_slot_idx(test)

print(f"Join 후 train shape: {train.shape}")
print(f"slot_idx 분포: {train['slot_idx'].value_counts().sort_index().head(3).to_dict()}")

Join 후 train shape: (250000, 109)
slot_idx 분포: {0: 10000, 1: 10000, 2: 10000}


## 4. 피처 엔지니어링분석에서 도출된 4개 Tier의 피처를 생성합니다:- **Tier 1**: 임계점 dummy + Binary indicator + 비율 변수- **Tier 2**: 시간 상호작용 + Cascading 곱항 + 누적 + Lag- **Tier 3**: 시나리오 메타 (★ 가장 강력)- **Layout**: layout_info의 새 비율과 상호작용

In [4]:
def build_features(df):
    """분석에서 도출된 모든 피처 생성. train과 test에 동일하게 적용"""
    
    # === Tier 1: 임계점 dummy ===
    df['pack_saturated'] = (df['pack_utilization'] > 0.95).astype(int)
    df['pack_idle_anomaly'] = ((df['pack_utilization'] < 0.1) & (df['slot_idx'] > 5)).astype(int)
    df['low_battery_alert'] = (df['low_battery_ratio'] > 0.034).astype(int)
    df['battery_critical'] = (df['battery_mean'] < 58).astype(int)
    df['density_alert'] = (df['max_zone_density'] > 0.05).astype(int)
    
    # Binary indicators (대부분 0인 변수들)
    df['has_collision'] = (df['near_collision_15m'] > 0).astype(int)
    df['has_block'] = (df['blocked_path_15m'] > 0).astype(int)
    df['has_fault'] = (df['fault_count_15m'] > 0).astype(int)
    df['has_reassign'] = (df['task_reassign_15m'] > 0).astype(int)
    
    # === 비율 변수 (★ 단일 변수 중 가장 강한 신호) ===
    df['total_fleet'] = df['robot_active'] + df['robot_idle'] + df['robot_charging']
    df['idle_ratio'] = df['robot_idle'] / (df['total_fleet'] + 1e-6)
    df['charging_ratio'] = df['robot_charging'] / (df['total_fleet'] + 1e-6)
    df['active_ratio'] = df['robot_active'] / (df['total_fleet'] + 1e-6)
    
    # === Tier 2: 시간 상호작용 ===
    df['pack_x_slot'] = df['pack_utilization'] * df['slot_idx']
    df['order_x_slot'] = df['order_inflow_15m'] * df['slot_idx']
    df['inflow_x_urgent'] = df['order_inflow_15m'] * df['urgent_order_ratio']
    df['effective_load'] = df['order_inflow_15m'] * (1 + 0.5 * df['urgent_order_ratio'])
    
    # === Cascading 곱항 ===
    df['pack_x_battery'] = df['pack_utilization'] * df['low_battery_ratio']
    df['pack_x_charging'] = df['pack_utilization'] * df['charging_ratio']
    df['inflow_x_battery'] = df['order_inflow_15m'] * df['low_battery_ratio']
    df['cong_x_inflow'] = df['congestion_score'] * df['order_inflow_15m']
    
    # === 누적 변수 (Stock variable) ===
    df['cum_orders'] = df.groupby('scenario_id')['order_inflow_15m'].cumsum()
    df['cum_collisions'] = df.groupby('scenario_id')['near_collision_15m'].cumsum()
    df['cum_faults'] = df.groupby('scenario_id')['fault_count_15m'].cumsum()
    df['cum_unique_sku'] = df.groupby('scenario_id')['unique_sku_15m'].cumsum()
    
    # === Lag 피처 ===
    for f in ['order_inflow_15m', 'pack_utilization', 'low_battery_ratio', 'congestion_score']:
        df[f'{f}_lag1'] = df.groupby('scenario_id')[f].shift(1)
        df[f'{f}_diff1'] = df[f] - df[f'{f}_lag1']
    
    # === 결측 플래그 ===
    for col in ['battery_mean', 'low_battery_ratio', 'congestion_score',
                'pack_utilization', 'order_inflow_15m']:
        df[f'{col}_isna'] = df[col].isna().astype(int)
    
    # === Layout 피처 ===
    df['robots_per_pack'] = df['robot_total'] / df['pack_station_count']
    df['robots_per_charger'] = df['robot_total'] / df['charger_count']
    df['area_per_robot'] = df['floor_area_sqm'] / df['robot_total']
    df['pack_per_charger'] = df['pack_station_count'] / df['charger_count']
    
    # Layout type 상호작용 (좁은 창고에서 배터리 위험)
    df['is_narrow'] = (df['layout_type'] == 'narrow').astype(int)
    df['narrow_x_battery'] = df['is_narrow'] * df['low_battery_ratio']
    df['few_packs'] = (df['pack_station_count'] < 7).astype(int)
    df['few_packs_x_pack_util'] = df['few_packs'] * df['pack_utilization']
    
    return df

train = build_features(train)
test = build_features(test)
print(f"피처 엔지니어링 후 shape: train={train.shape}, test={test.shape}")

피처 엔지니어링 후 shape: train=(250000, 155), test=(50000, 154)


## 5. 시나리오 메타 피처 (★★★ 가장 강력)분석에서 발견한 가장 중요한 사실:> **between-scenario 변동이 within-scenario 변동의 3.3배**각 시나리오의 25개 슬롯에서 통계량(mean/std/max/min)을 계산해서 모든 슬롯에 broadcast.이게 모델 Top 피처의 대부분을 차지합니다.

In [5]:
# 시나리오 단위 집계 - 가장 강력한 피처들
key_features = [
    'order_inflow_15m', 'pack_utilization', 'low_battery_ratio',
    'congestion_score', 'urgent_order_ratio', 'idle_ratio', 'charging_ratio',
    'robot_charging', 'fault_count_15m', 'staff_on_floor',
]

def add_scenario_meta(df, features):
    for f in features:
        sc_agg = df.groupby('scenario_id')[f].agg(['mean', 'std', 'max', 'min'])
        sc_agg.columns = [f'{f}_sc_{stat}' for stat in sc_agg.columns]
        df = df.merge(sc_agg, on='scenario_id', how='left')
    return df

train = add_scenario_meta(train, key_features)
test = add_scenario_meta(test, key_features)

print(f"최종 shape: train={train.shape}, test={test.shape}")
print(f"피처 개수: {train.shape[1] - 4}")  # ID, layout_id, scenario_id, target 제외

최종 shape: train=(250000, 195), test=(50000, 194)
피처 개수: 191


## 6. 학습 준비- 카테고리형 변환: `layout_id`, `layout_type`- 타깃 log1p 변환: heavy-tail 완화 (skew 5.68 → 0.08)- GroupKFold 그룹 정의: scenario_id

In [6]:
# meta_cols에서 layout_id 제외 (피처로 사용)
meta_cols = ['ID', 'scenario_id', target_col]  # layout_id 제거
feature_cols = [c for c in train.columns if c not in meta_cols]
print(f"피처 수: {len(feature_cols)}")

# 카테고리형 변환
train['layout_id'] = train['layout_id'].astype('category')
train['layout_type'] = train['layout_type'].astype('category')
test['layout_id'] = test['layout_id'].astype('category')
test['layout_type'] = test['layout_type'].astype('category')

# 학습 데이터
X_train = train[feature_cols]
y_train = train[target_col].values
y_log = np.log1p(y_train)
groups = train['scenario_id'].values

X_test = test[feature_cols]

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"\ncategorical 컬럼 자동 감지:")
print(f"  layout_id   : {X_train['layout_id'].dtype}")
print(f"  layout_type : {X_train['layout_type'].dtype}")

피처 수: 192
X_train: (250000, 192)
X_test:  (50000, 192)

categorical 컬럼 자동 감지:
  layout_id   : category
  layout_type : category


## 7. LightGBM 5-fold GroupKFold 학습**중요한 설정**:- `GroupKFold` by `scenario_id`: 시나리오 누설 방지 (★ 절대 깨지 말 것)- `log1p` 타깃: heavy-tail 처리- `categorical_feature` 지정: layout_id, layout_type

In [7]:
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'min_child_samples': 100,
    'feature_fraction': 0.7,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'random_state': 42,
    'verbose': -1,
    'n_jobs': -1,
}

gkf = GroupKFold(n_splits=5)
oof_pred_log = np.zeros(len(X_train))
test_pred_log = np.zeros(len(X_test))
fold_scores = []
feature_importance = pd.DataFrame()

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_train, y_log, groups=groups)):
    print(f"\n{'='*60}\nFold {fold+1}/5\n{'='*60}")
    
    X_tr = X_train.iloc[tr_idx]
    X_val = X_train.iloc[val_idx]
    y_tr_log = y_log[tr_idx]
    y_val_log = y_log[val_idx]
    y_val_raw = y_train[val_idx]
    
    # categorical_feature 인자 제거 - pandas category dtype이 자동 인식됨
    train_data = lgb.Dataset(X_tr, label=y_tr_log)
    val_data = lgb.Dataset(X_val, label=y_val_log, reference=train_data)
    
    model = lgb.train(
        params, train_data,
        num_boost_round=3000,
        valid_sets=[val_data],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)]
    )
    
    # OOF 저장
    oof_pred_log[val_idx] = model.predict(X_val)
    
    # Test 예측 (5 fold 평균)
    test_pred_log += model.predict(X_test) / 5
    
    # 평가
    pred_raw = np.clip(np.expm1(oof_pred_log[val_idx]), 0, None)
    rmse = np.sqrt(mean_squared_error(y_val_raw, pred_raw))
    mae = mean_absolute_error(y_val_raw, pred_raw)
    fold_scores.append({'fold': fold, 'rmse': rmse, 'mae': mae, 
                        'best_iter': model.best_iteration})
    print(f"Fold {fold} -> RMSE: {rmse:.4f}, MAE: {mae:.4f}, "
          f"best_iter: {model.best_iteration}")
    
    # Feature importance 누적
    imp = pd.DataFrame({
        'feature': model.feature_name(),
        'gain': model.feature_importance(importance_type='gain'),
    })
    feature_importance = pd.concat([feature_importance, imp])


Fold 1/5
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[95]	valid_0's rmse: 0.636565
Fold 0 -> RMSE: 23.6201, MAE: 8.8656, best_iter: 95

Fold 2/5
Training until validation scores don't improve for 100 rounds
[200]	valid_0's rmse: 0.656817
Early stopping, best iteration is:
[188]	valid_0's rmse: 0.656709
Fold 1 -> RMSE: 23.0569, MAE: 9.1447, best_iter: 188

Fold 3/5
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[88]	valid_0's rmse: 0.637576
Fold 2 -> RMSE: 21.2229, MAE: 8.6021, best_iter: 88

Fold 4/5
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[87]	valid_0's rmse: 0.667508
Fold 3 -> RMSE: 24.5331, MAE: 9.4261, best_iter: 87

Fold 5/5
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[92]	valid_0's rmse: 0.647968
Fold 4 -> RMSE: 22.0647, MAE: 8.9600, best_iter: 92


## 8. 최종 평가OOF 예측을 raw scale로 복원하고 전체 RMSE/MAE를 계산합니다.

In [8]:
# 최종 예측 (log → raw 복원)
oof_pred = np.clip(np.expm1(oof_pred_log), 0, None)
test_pred = np.clip(np.expm1(test_pred_log), 0, None)

overall_rmse = np.sqrt(mean_squared_error(y_train, oof_pred))
overall_mae = mean_absolute_error(y_train, oof_pred)

fold_df = pd.DataFrame(fold_scores)
print(f"=== 최종 결과 ===")
print(f"OOF RMSE: {overall_rmse:.4f}")
print(f"OOF MAE : {overall_mae:.4f}")
print(f"\nFold std: RMSE±{fold_df['rmse'].std():.4f}, MAE±{fold_df['mae'].std():.4f}")
print(f"\nFold별 결과:")
print(fold_df)

=== 최종 결과 ===
OOF RMSE: 22.9289
OOF MAE : 8.9997

Fold std: RMSE±1.2962, MAE±0.3084

Fold별 결과:
   fold       rmse       mae  best_iter
0     0  23.620137  8.865555         95
1     1  23.056887  9.144652        188
2     2  21.222861  8.602141         88
3     3  24.533104  9.426122         87
4     4  22.064685  8.959994         92


## 9. 잔차 진단 (target 구간별 MAE)**Burst 영역(60분+)이 RMSE의 주된 source**입니다. 평균 영역은 어떤 모델도 잘 맞춥니다.

In [9]:
diag = pd.DataFrame({'y_true': y_train, 'y_pred': oof_pred})
diag['abs_err'] = np.abs(diag['y_true'] - diag['y_pred'])
diag['target_bin'] = pd.cut(diag['y_true'], 
                              bins=[-1, 5, 15, 30, 60, 120, 1000],
                              labels=['0-5', '5-15', '15-30', '30-60', '60-120', '120+'])

bin_stats = diag.groupby('target_bin', observed=True).agg(
    count=('y_true', 'count'),
    mean_y=('y_true', 'mean'),
    mae=('abs_err', 'mean'),
    p90_err=('abs_err', lambda x: x.quantile(0.9))
).round(2)
print(bin_stats)

            count  mean_y     mae  p90_err
target_bin                                
0-5         75049    2.73    2.59     4.62
5-15        82118    8.66    3.83     7.74
15-30       40732   22.07    9.22    17.28
30-60       39186   41.35   11.74    24.88
60-120      10370   78.31   47.18    75.36
120+         2545  193.66  163.64   269.53


## 10. Top 30 피처 중요도시나리오 메타(_sc_*)와 비율 변수(idle_ratio, charging_ratio)가 상위에 있을 것.

In [10]:
imp_summary = (feature_importance.groupby('feature')['gain']
                .sum().sort_values(ascending=False).head(30))

print("Top 30 피처 (gain 합계 기준):")
for i, (feat, gain) in enumerate(imp_summary.items(), 1):
    # 카테고리 표시
    if '_sc_' in feat:
        marker = ' [Scenario Meta]'
    elif feat in ['idle_ratio', 'charging_ratio', 'active_ratio', 'total_fleet']:
        marker = ' [Ratio]'
    elif '_x_' in feat or feat == 'effective_load':
        marker = ' [Interaction]'
    elif feat.startswith('cum_'):
        marker = ' [Stock]'
    elif '_lag' in feat or '_diff' in feat:
        marker = ' [Lag]'
    elif feat.startswith('has_') or feat.endswith(('_alert', '_critical', '_saturated')):
        marker = ' [Dummy]'
    elif feat in layout.columns or feat.startswith(('robots_per', 'area_per', 'pack_per',
                                                     'is_narrow', 'narrow_x', 'few_packs')):
        marker = ' [Layout]'
    else:
        marker = ''
    print(f"  {i:2d}. {feat:42s}: {gain:>12,.0f}{marker}")

Top 30 피처 (gain 합계 기준):
   1. charging_ratio_sc_mean                    :    1,193,292 [Scenario Meta]
   2. congestion_score_sc_mean                  :    1,000,545 [Scenario Meta]
   3. layout_id                                 :      789,367 [Layout]
   4. robot_charging_sc_mean                    :      649,154 [Scenario Meta]
   5. low_battery_ratio_sc_mean                 :      644,154 [Scenario Meta]
   6. pack_utilization_sc_mean                  :      643,454 [Scenario Meta]
   7. low_battery_ratio_sc_std                  :      374,762 [Scenario Meta]
   8. charging_ratio                            :      152,312 [Ratio]
   9. pack_utilization_sc_min                   :      141,497 [Scenario Meta]
  10. congestion_score_sc_std                   :      134,966 [Scenario Meta]
  11. charging_ratio_sc_max                     :      133,980 [Scenario Meta]
  12. pack_utilization                          :      133,334
  13. order_inflow_15m_sc_mean                  :       95,

## 11. 제출 파일 생성검증 후 `submission.csv`로 저장합니다.

In [11]:
submission = pd.DataFrame({
    'ID': test['ID'],
    target_col: test_pred
})

# 검증
assert len(submission) == len(test), "행 수 불일치"
assert submission[target_col].notna().all(), "NaN 존재"
assert (submission[target_col] >= 0).all(), "음수 존재"

submission.to_csv('submission.csv', index=False)

print(f"=== submission.csv 저장 완료 ===")
print(f"행 수: {len(submission)}")
print(f"\n예측 통계:")
print(f"  min:    {test_pred.min():.2f}")
print(f"  max:    {test_pred.max():.2f}")
print(f"  mean:   {test_pred.mean():.2f}")
print(f"  median: {np.median(test_pred):.2f}")
print(f"\n첫 5행:")
print(submission.head())

# OOF/Test 예측 저장 (앙상블/분석용)
np.save('oof_pred.npy', oof_pred)
np.save('test_pred.npy', test_pred)
print("\noof_pred.npy, test_pred.npy 저장 완료")

=== submission.csv 저장 완료 ===
행 수: 50000

예측 통계:
  min:    0.00
  max:    81.21
  mean:   17.72
  median: 12.64

첫 5행:
            ID  avg_delay_minutes_next_30m
0  TEST_015375                   15.847047
1  TEST_015376                   23.113920
2  TEST_015377                   28.595146
3  TEST_015378                   29.283258
4  TEST_015379                   29.917764

oof_pred.npy, test_pred.npy 저장 완료


---## 12. (선택) 분위수 회귀 + 앙상블추가 1~3% 개선이 필요하면 이 부분을 실행하세요.3개 모델 학습 후 가중 앙상블:- **Mean (RMSE objective)**: 기본 예측- **P50 (Quantile alpha=0.5)**: median 예측, 다른 손실함수- **P90 (Quantile alpha=0.9)**: 큰 값 예측 특화 (burst 보강)

In [15]:
def train_quantile(alpha):
    """분위수 회귀 학습 (LightGBM의 같은 fold 사용)"""
    params_q = {**params, 'objective': 'quantile', 'alpha': alpha, 'metric': 'quantile'}
    oof = np.zeros(len(X_train))
    test_p = np.zeros(len(X_test))
    
    for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_train, y_log, groups=groups)):
        print(f"  alpha={alpha} - Fold {fold+1}", end=' ')
        X_tr = X_train.iloc[tr_idx]
        X_val = X_train.iloc[val_idx]
        y_tr_log = y_log[tr_idx]
        y_val_log = y_log[val_idx]
        
        # categorical_feature 인자 제거 (pandas category dtype 자동 인식)
        train_data = lgb.Dataset(X_tr, label=y_tr_log)
        val_data = lgb.Dataset(X_val, label=y_val_log, reference=train_data)
        
        model = lgb.train(params_q, train_data, num_boost_round=3000,
                          valid_sets=[val_data],
                          callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
        oof[val_idx] = model.predict(X_val)
        test_p += model.predict(X_test) / 5
        print(f"iter={model.best_iteration}")
    
    return np.clip(np.expm1(oof), 0, None), np.clip(np.expm1(test_p), 0, None)

# 시간 오래 걸림 (각 alpha별 5-fold)
print("학습 중... (시간 오래 걸림)\n")
print("[P50 학습]")
oof_p50, test_p50 = train_quantile(0.5)
print("\n[P90 학습]")  
oof_p90, test_p90 = train_quantile(0.9)

print(f"\n개별 OOF:")
print(f"  Mean: RMSE={np.sqrt(mean_squared_error(y_train, oof_pred)):.4f}, "
      f"MAE={mean_absolute_error(y_train, oof_pred):.4f}")
print(f"  P50:  RMSE={np.sqrt(mean_squared_error(y_train, oof_p50)):.4f}, "
      f"MAE={mean_absolute_error(y_train, oof_p50):.4f}")
print(f"  P90:  RMSE={np.sqrt(mean_squared_error(y_train, oof_p90)):.4f}, "
      f"MAE={mean_absolute_error(y_train, oof_p90):.4f}")

학습 중... (시간 오래 걸림)

[P50 학습]
  alpha=0.5 - Fold 1 Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[541]	valid_0's quantile: 0.227904
iter=541
  alpha=0.5 - Fold 2 Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[405]	valid_0's quantile: 0.234728
iter=405
  alpha=0.5 - Fold 3 Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[518]	valid_0's quantile: 0.228483
iter=518
  alpha=0.5 - Fold 4 Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[352]	valid_0's quantile: 0.235148
iter=352
  alpha=0.5 - Fold 5 Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[155]	valid_0's quantile: 0.232023
iter=155

[P90 학습]
  alpha=0.9 - Fold 1 Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[88]	valid_0's quantile: 0.10083


## 13. 가중 앙상블 grid searchOOF 데이터로 최적 가중치 탐색

In [16]:
# RMSE 최소화 가중치 탐색
best_rmse = float('inf')
best_w_rmse = None
for w_mean in np.arange(0.4, 0.91, 0.05):
    for w_p50 in np.arange(0.0, 1-w_mean+0.01, 0.05):
        w_p90 = 1 - w_mean - w_p50
        if w_p90 < 0: continue
        ens = w_mean * oof_pred + w_p50 * oof_p50 + w_p90 * oof_p90
        rmse = np.sqrt(mean_squared_error(y_train, ens))
        if rmse < best_rmse:
            best_rmse = rmse
            best_mae_at_rmse = mean_absolute_error(y_train, ens)
            best_w_rmse = (w_mean, w_p50, w_p90)

# MAE 최소화 가중치 탐색
best_mae = float('inf')
best_w_mae = None
for w_mean in np.arange(0.4, 0.91, 0.05):
    for w_p50 in np.arange(0.0, 1-w_mean+0.01, 0.05):
        w_p90 = 1 - w_mean - w_p50
        if w_p90 < 0: continue
        ens = w_mean * oof_pred + w_p50 * oof_p50 + w_p90 * oof_p90
        mae = mean_absolute_error(y_train, ens)
        if mae < best_mae:
            best_mae = mae
            best_rmse_at_mae = np.sqrt(mean_squared_error(y_train, ens))
            best_w_mae = (w_mean, w_p50, w_p90)

print(f"=== RMSE 최소화 ===")
print(f"  Weights (mean, p50, p90): {best_w_rmse}")
print(f"  RMSE: {best_rmse:.4f}, MAE: {best_mae_at_rmse:.4f}")

print(f"\n=== MAE 최소화 ===")
print(f"  Weights (mean, p50, p90): {best_w_mae}")
print(f"  RMSE: {best_rmse_at_mae:.4f}, MAE: {best_mae:.4f}")

# 메트릭이 뭔지 모르므로 둘 다 저장
# 일반적으로 MAE 최적 가중치 사용 (1등이 9.7이면 MAE일 가능성)
chosen_w = best_w_mae

test_ens = (chosen_w[0] * test_pred + 
            chosen_w[1] * test_p50 + 
            chosen_w[2] * test_p90)

submission_ens = pd.DataFrame({'ID': test['ID'], target_col: test_ens})
submission_ens.to_csv('submission_ensemble.csv', index=False)
print(f"\nsubmission_ensemble.csv 저장 (weights: {chosen_w})")

=== RMSE 최소화 ===
  Weights (mean, p50, p90): (np.float64(0.4), np.float64(0.25), np.float64(0.35))
  RMSE: 21.0716, MAE: 9.6877

=== MAE 최소화 ===
  Weights (mean, p50, p90): (np.float64(0.5), np.float64(0.45), np.float64(0.04999999999999999))
  RMSE: 22.2708, MAE: 8.9361

submission_ensemble.csv 저장 (weights: (np.float64(0.5), np.float64(0.45), np.float64(0.04999999999999999)))


In [18]:
sub = pd.read_csv('submission.csv')
sub = sub.sort_values('ID').reset_index(drop=True)
sub.to_csv('submission.csv', index=False)

sub_ensemble = pd.read_csv('submission_ensemble.csv')
sub_ensemble = sub_ensemble.sort_values('ID').reset_index(drop=True)
sub_ensemble.to_csv('submission_ensemble.csv', index=False)